In [2]:
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import precision_recall_fscore_support
from nltk.corpus import stopwords, opinion_lexicon
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from scipy.sparse import hstack
import nltk
import numpy as np

# Download the packages
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('opinion_lexicon')

# Set global variables
raw_data = []  # Data from the file
train_data = []  # Preprocessed training data
test_data = []  # Preprocessed test data
global_feature_dict = {}  # Global feature dictionary

# Load the data
def load_data():
    """Load data from the specified CSV file and append it to raw_data."""
    global raw_data
    data_file_path = r"SourceFiles/sentiment-dataset.csv"  # Correct file path
    
    dataset = pd.read_csv(data_file_path, header=None, sep='\t', names=['Tweet ID', 'Sentiment', 'Tweet Text'])
    dataset.columns = dataset.columns.str.strip()
    
    for index, row in dataset.iterrows():
        label = row['Sentiment']
        text = row['Tweet Text']
        raw_data.append((text, label))

# Preprocessing the data
def pre_process(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|@[\w]+|#\S+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    tokens = word_tokenize(text)
    stop_words = set(stopwords.words('english'))
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]
    return tokens

# Convert tokens to feature vectors
def to_feature_vector(tokens):
    return {token: 1 for token in tokens}

# Extract stylistic features
def extract_stylistic_features(text):
    length = len(text.split())
    punctuation_count = sum([1 for char in text if char in ['.', '!', '?']])
    return [length, punctuation_count]

# Lexicon-based features
positive_words = set(opinion_lexicon.positive())
negative_words = set(opinion_lexicon.negative())

def lexicon_features(text):
    words = set(text.split())
    positive_count = len(words & positive_words)
    negative_count = len(words & negative_words)
    return [positive_count, negative_count]

# TF-IDF feature extraction
def extract_tfidf_features(raw_data):
    tfidf_vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1,2), sublinear_tf=True, max_df=0.9, min_df=3)
    X_tfidf = tfidf_vectorizer.fit_transform([text for text, _ in raw_data])
    return X_tfidf, tfidf_vectorizer

# Prepare combined features
def prepare_combined_features(raw_data):
    X_tfidf, tfidf_vectorizer = extract_tfidf_features(raw_data)
    numeric_feats = np.array([
        extract_stylistic_features(text) + lexicon_features(text)
        for text, _ in raw_data
    ])
    X_combined = hstack([X_tfidf, numeric_feats])
    y = np.array([label for _, label in raw_data])
    return X_combined, y, tfidf_vectorizer

# Train and evaluate model
def train_evaluate_model(X, y):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    param_grid = {
        'C': [1, 2, 5, 10],
        'class_weight': [None, 'balanced'],
        'max_iter': [10000]
    }
    grid = GridSearchCV(LinearSVC(), param_grid, cv=5, scoring='f1_weighted', n_jobs=-1)
    grid.fit(X_train, y_train)
    print("Best parameters:", grid.best_params_)
    
    clf = LinearSVC(**grid.best_params_)
    clf.fit(X_train, y_train)
    
    preds = clf.predict(X_test)
    precision, recall, f1, _ = precision_recall_fscore_support(y_test, preds, average='weighted')
    print(f"Precision: {precision:.3f}, Recall: {recall:.3f}, F1 Score: {f1:.3f}")
    return clf

# MAIN EXECUTION
load_data()
X_combined, y, tfidf_vectorizer = prepare_combined_features(raw_data)
final_model = train_evaluate_model(X_combined, y)


[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/anvithasubramani/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/anvithasubramani/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     /Users/anvithasubramani/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/anvithasubramani/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package opinion_lexicon to
[nltk_data]     /Users/anvithasubramani/nltk_data...
[nltk_data]   Package opinion_lexicon is already up-to-date!
/Users/anvithasubramani/Documents/Source/Assignment1/Q4/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/anvithas

Best parameters: {'C': 1, 'class_weight': None, 'max_iter': 10000}
Precision: 0.863, Recall: 0.864, F1 Score: 0.863
